# Alimentation de la fiche produit NBO

Un bloc de code par section du classeur Excel. Chaque bloc affiche les valeurs
à recopier dans les cellules jaunes de la feuille produit.

**Ne pas automatiser** : les KPI d'équipement du profiling (multi-équipé,
souscription 12 mois) — à saisir manuellement.


## 0 — Configuration


In [ ]:
import pandas as pd, numpy as np
from scipy.stats import spearmanr

# ---- Colonnes ----
ID    = "Identifiant Tiers"
SCORE = "score"
RECO  = "recommandation"
DATE_SOUS = "date_souscription"
CLIC  = "clic"

# ---- Millésimes : nom -> (fichier scores, date de photo, fichier base éligible) ----
MILLESIMES = {
    "Sep 2025":  ("scores_sep2025.csv",  "2025-09-30", "base_sep2025.xlsx"),
    "Mars 2026": ("scores_mars2026.csv", "2026-03-31", "base_mars2026.xlsx"),
    "Juin 2026": ("scores_juin2026.csv", "2026-06-30", "base_juin2026.xlsx"),
}
REF, CUR = "Mars 2026", "Juin 2026"      # millésimes du PSI (section 4)

# ---- Détenteurs : ID + date de souscription ----
F_DETENTEURS = "detenteurs.csv"

# ---- Campagne ----
F_CAMPAGNE   = "campagne.csv"            # ID + colonne clic (1/0)
MOIS_OC      = "2026-03"                 # mois de la campagne
FENETRE_MOIS = 3                         # fenêtre d'observation
MIL_CAMPAGNE = "Sep 2025"                # millésime de score antérieur au lancement

# ---- Profiling : colonnes du fichier base éligible ----
COLS_PROFIL = {"Âge moyen (ans)": "age",
               "Ancienneté de la relation (ans)": "anciennete",
               "Part de clients actifs": "actif_3m",
               "Part de clients Mass Affluent": "mass_affluent"}
FILTRE_PARC = None                       # ex. "age < 25" pour Pack Jeunes, sinon None

# ---- Seuil : déduit du flag recommandation (plus petit score recommandé) ----
SEUIL_MANUEL = {}                        # ex. {"Sep 2025": 0.157} pour forcer une valeur


In [ ]:
def lire(f):
    return pd.read_excel(f) if str(f).endswith((".xlsx", ".xls")) else pd.read_csv(f)

# Scores + bases par millésime
scores, bases, dates, seuils = {}, {}, {}, {}
for m, (f_sc, dt, f_base) in MILLESIMES.items():
    d = lire(f_sc)
    d[SCORE] = pd.to_numeric(d[SCORE], errors="coerce")
    scores[m] = d
    bases[m]  = lire(f_base)
    dates[m]  = pd.Timestamp(dt)
    seuils[m] = SEUIL_MANUEL.get(m, d.loc[d[RECO] == 1, SCORE].min())

# Détenteurs
det = lire(F_DETENTEURS)
det[DATE_SOUS] = pd.to_datetime(det[DATE_SOUS], errors="coerce")

print("--- Contrôle des clés ---")
for m in MILLESIMES:
    n_sc, n_ba = len(scores[m]), len(bases[m])
    inter = scores[m][ID].isin(bases[m][ID]).sum()
    print(f"{m:10} | scorés {n_sc:>7,} | base {n_ba:>7,} | scorés présents en base : {inter/n_sc:.1%}")
print(f"\nDétenteurs : {len(det):,} | présents en base ({CUR}) : "
      f"{det[ID].isin(bases[CUR][ID]).mean():.1%}")
print("\nSeuils déduits :", {m: round(s, 4) for m, s in seuils.items()})


## Sections 2 et 3 — Volumétrie, seuils et couverture


In [ ]:
lignes = []
for m in MILLESIMES:
    d, s = scores[m], seuils[m]
    det_m = det[det[DATE_SOUS] <= dates[m]][ID]          # détenteurs à la date de photo
    lignes.append({
        "Millésime": m,
        "Seuil": round(s, 4),
        "Clients scorés": len(d),
        "Score = 0": int((d[SCORE] == 0).sum()),
        "0 < Score < Seuil": int(((d[SCORE] > 0) & (d[SCORE] < s)).sum()),
        "Score >= Seuil": int((d[SCORE] >= s).sum()),
        "Vrais détenteurs": int(det_m.nunique()),
        "Base éligible": len(bases[m]),
    })
vol = pd.DataFrame(lignes)
vol["Contrôle"] = np.where(
    vol[["Score = 0", "0 < Score < Seuil", "Score >= Seuil"]].sum(axis=1) == vol["Clients scorés"],
    "OK", "ÉCART")

print("--- Section 2 (B10:G12) ---")
print(vol.drop(columns=["Base éligible"]).to_string(index=False))

print("\n--- Section 3 (B16:D20) ---")
sec3 = pd.DataFrame({
    "Base éligible": vol["Base éligible"].values,
    "Taux de couverture": (vol["Clients scorés"] / vol["Base éligible"]).values,
    "Part de scores nuls": (vol["Score = 0"] / vol["Clients scorés"]).values,
    "Taux de sélection": (vol["Score >= Seuil"] / vol["Clients scorés"]).values,
    "Taux de détention": (vol["Vrais détenteurs"] / vol["Base éligible"]).values,
}, index=list(vol["Millésime"])).T
print(sec3.to_string(float_format=lambda x: f"{x:,.4f}"))


## Section 4 — PSI (déciles figés sur le millésime de référence)


In [ ]:
s_ref = scores[REF].loc[scores[REF][SCORE] > 0, SCORE]     # zéros exclus
s_cur = scores[CUR].loc[scores[CUR][SCORE] > 0, SCORE]

bornes = np.unique(np.quantile(s_ref, np.linspace(0, 1, 11)))
bornes[0], bornes[-1] = 0, np.inf

e_ref = pd.cut(s_ref, bornes).value_counts().sort_index()
e_cur = pd.cut(s_cur, bornes).value_counts().sort_index().reindex(e_ref.index, fill_value=0)

psi = pd.DataFrame({"eff_ref": e_ref.values, "eff_cur": e_cur.values,
                    "bornes": e_ref.index.astype(str)})
psi["pct_ref"] = psi.eff_ref / psi.eff_ref.sum()
psi["pct_cur"] = psi.eff_cur / psi.eff_cur.sum()
psi["contrib"] = (psi.pct_cur - psi.pct_ref) * np.log(
    psi.pct_cur.clip(lower=1e-6) / psi.pct_ref.clip(lower=1e-6))
psi = psi.iloc[::-1].reset_index(drop=True); psi.index += 1   # tranche 1 = meilleurs scores

if len(psi) < 10:
    print(f"ATTENTION : {len(psi)} tranches au lieu de 10 — masse de scores identiques "
          f"({s_ref.value_counts().iloc[0]:,} clients sur la valeur la plus fréquente).\n")

print(psi.round(4).to_string())
print(f"\nPSI total : {psi.contrib.sum():.4f}")
print(f"\n--- À coller en B32:C41 ---\n"
      f"{psi[['eff_ref','eff_cur']].to_string(index=False, header=False)}")


## Section 5 — Stabilité de la liste à cibler


In [ ]:
mils = list(MILLESIMES)
prec = mils[mils.index(CUR) - 1]

liste_cur  = set(scores[CUR].loc[scores[CUR][SCORE] >= seuils[CUR], ID])
liste_prec = set(scores[prec].loc[scores[prec][SCORE] >= seuils[prec], ID])

j = (scores[prec][[ID, SCORE]].rename(columns={SCORE: "s_prec"})
     .merge(scores[CUR][[ID, SCORE]].rename(columns={SCORE: "s_cur"}), on=ID))
j = j[(j.s_prec > 0) & (j.s_cur > 0)]
rho = spearmanr(j.s_prec, j.s_cur).correlation

print(f"Nb clients >= seuil ({CUR})            : {len(liste_cur):,}")
print(f"dont déjà >= seuil ({prec})            : {len(liste_cur & liste_prec):,}")
print(f"Taux de recouvrement                    : {len(liste_cur & liste_prec)/len(liste_cur):.1%}")
print(f"Spearman ({prec} vs {CUR})              : {rho:.2f}  sur {len(j):,} clients communs")


## Section 6 — Contrôle détenteurs


In [ ]:
det_cur = det[det[DATE_SOUS] <= dates[CUR]][[ID]].drop_duplicates()
d = det_cur.merge(scores[CUR][[ID, SCORE]], on=ID, how="left")
s = seuils[CUR]

n_tot   = len(d)
n_fuite = int((d[SCORE] >= s).sum())
n_zero  = int((d[SCORE] == 0).sum())
n_entre = int(((d[SCORE] > 0) & (d[SCORE] < s)).sum())
n_abs   = int(d[SCORE].isna().sum())

print(f"Nb de vrais détenteurs ({CUR})   : {n_tot:,}")
print(f"dont score >= seuil (fuite)      : {n_fuite:,}  ({n_fuite/n_tot:.1%})")
print(f"dont score = 0                   : {n_zero:,}  ({n_zero/n_tot:.1%})")
print(f"dont 0 < score < seuil           : {n_entre:,}")
print(f"dont absents du fichier de scores: {n_abs:,}  ({n_abs/n_tot:.1%})   <-- à surveiller")


## Section 7 — Backtest campagne


In [ ]:
camp = lire(F_CAMPAGNE)
fin_camp = pd.Timestamp(MOIS_OC) + pd.offsets.MonthEnd(0)
fin_obs  = fin_camp + pd.DateOffset(months=FENETRE_MOIS)

# Souscripteurs = détenteurs dont la souscription tombe dans la fenêtre
sous = set(det.loc[det[DATE_SOUS].between(fin_camp, fin_obs, inclusive="right"), ID])

s = seuils[MIL_CAMPAGNE]
c = camp.merge(scores[MIL_CAMPAGNE][[ID, SCORE]], on=ID, how="left")
c["souscripteur"] = c[ID].isin(sous).astype(int)
c["tranche"] = np.select(
    [c[SCORE] == 0, (c[SCORE] > 0) & (c[SCORE] < s), c[SCORE] >= s],
    ["Score = 0", "0 < Score < Seuil", "Score >= Seuil"], default="Non scoré")

print(f"Campagne close le {fin_camp:%d/%m/%Y} — observation jusqu'au {fin_obs:%d/%m/%Y}")
print(f"Liste campagne : {len(c):,} | dans la liste modèle (>= seuil) : "
      f"{(c.tranche=='Score >= Seuil').sum():,} "
      f"({(c.tranche=='Score >= Seuil').mean():.1%})")
if CLIC in c.columns:
    print(f"Cliqueurs : {c[CLIC].sum():,} | cliqueurs souscripteurs : "
          f"{int(((c[CLIC]==1) & (c.souscripteur==1)).sum()):,}")

print("\n--- 7.1 Conversion par tranche (B73:C75) ---")
t = (c.groupby("tranche").agg(ciblés=(ID, "size"), souscripteurs=("souscripteur", "sum"))
     .reindex(["Score = 0", "0 < Score < Seuil", "Score >= Seuil", "Non scoré"]).fillna(0).astype(int))
t["conversion"] = t.souscripteurs / t.ciblés
t.loc["Total"] = [t.ciblés.sum(), t.souscripteurs.sum(), t.souscripteurs.sum() / t.ciblés.sum()]
print(t.to_string(formatters={"conversion": "{:.2%}".format}))

sel = t.loc["Score >= Seuil", "ciblés"] / t.loc["Total", "ciblés"]
cap = t.loc["Score >= Seuil", "souscripteurs"] / max(t.loc["Total", "souscripteurs"], 1)
print(f"\n--- 7.2 ---\nTaux de sélection : {sel:.1%} | Taux de capture : {cap:.1%} "
      f"| Lift : {cap/sel if sel else float('nan'):.2f} | N souscripteurs : {t.loc['Total','souscripteurs']:,}")


In [ ]:
# --- 7.3 Courbe de gains par décile (population campagne scorée) ---
cs = c[c[SCORE].notna()].copy()
cs["decile"] = pd.qcut(cs[SCORE].rank(method="first", ascending=False), 10, labels=range(1, 11))

g = cs.groupby("decile", observed=True).agg(ciblés=(ID, "size"), souscripteurs=("souscripteur", "sum"))
g["conversion"] = g.souscripteurs / g.ciblés
print(g.to_string(formatters={"conversion": "{:.2%}".format}))
print(f"\n--- À coller en B92:C101 ---\n"
      f"{g[['ciblés','souscripteurs']].to_string(index=False, header=False)}")


## Section 8 — Profiling


In [ ]:
base = bases[CUR]
if FILTRE_PARC:
    base = base.query(FILTRE_PARC)

cibles = base[base[ID].isin(scores[CUR].loc[scores[CUR][SCORE] >= seuils[CUR], ID])]
detent = base[base[ID].isin(det[det[DATE_SOUS] <= dates[CUR]][ID])]

prof = pd.DataFrame({
    "Clients à cibler": [cibles[col].mean() for col in COLS_PROFIL.values()],
    "Vrais détenteurs": [detent[col].mean() for col in COLS_PROFIL.values()],
    "Parc de comptes":  [base[col].mean()   for col in COLS_PROFIL.values()],
}, index=list(COLS_PROFIL))
prof["Écart cibles - parc"] = prof["Clients à cibler"] - prof["Parc de comptes"]

print(f"Effectifs — cibles : {len(cibles):,} | détenteurs : {len(detent):,} | parc : {len(base):,}")
print(prof.round(4).to_string())


## Backtest sans campagne — acquisitions depuis chaque millésime


In [ ]:
res = []
for m in ["Sep 2025", "Mars 2026"]:
    acq = det[det[DATE_SOUS] > dates[m]][[ID]].drop_duplicates()
    a = acq.merge(scores[m][[ID, SCORE]], on=ID, how="left")
    s = seuils[m]
    res.append({
        "Millésime": m,
        "Acquisitions": len(a),
        "Score >= seuil": int((a[SCORE] >= s).sum()),
        "Score < seuil":  int(((a[SCORE] < s) & a[SCORE].notna()).sum()),
        "Absents des scores": int(a[SCORE].isna().sum()),
    })
bt = pd.DataFrame(res).set_index("Millésime")
for col in ["Score >= seuil", "Score < seuil", "Absents des scores"]:
    bt[col + " %"] = bt[col] / bt["Acquisitions"]
print(bt.to_string(formatters={c: "{:.1%}".format for c in bt.columns if c.endswith("%")}))
